In [1]:
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

from nRC_parametrization import SOC_func
from nRC_parametrization import SOC_to_OCV
from nRC_parametrization import process_dynamic
from nRC_parametrization import simulate_battery_model
from nRC_parametrization import RMSE

In [2]:
def data_visualization(U, U_sim, OCV, I, t, SOC):
    plt.figure(figsize=(10, 8))

    #  Voltage
    plt.subplot(4, 1, 1)
    plt.plot(t, U, label='measured', linewidth=1.2)
    plt.plot(t, OCV, label='OCV', linewidth=1.2)
    plt.plot(t, U_sim, '--', label='simulated', linewidth=1.2)
    plt.ylabel('Voltage, V')
    plt.legend()
    plt.grid(True, alpha=0.3)

    #  Current
    plt.subplot(4, 1, 2)
    plt.plot(t, I, label='current', linewidth=1.2)
    plt.ylabel('Current, A')
    plt.title('Current')
    plt.grid(True, alpha=0.3)

    #  SOC
    plt.subplot(4, 1, 3)
    plt.plot(t, SOC, label='SOC', color='purple', linewidth=1.2)
    plt.ylabel('SOC')
    plt.title('State of Charge')
    plt.grid(True, alpha=0.3)

    # Error
    err = U - U_sim
    plt.subplot(4, 1, 4)
    plt.plot(t, err, label='error', linewidth=1.2)
    plt.ylabel('Error, V')
    plt.xlabel('Time, s')
    plt.title(f'Error (mean abs = {np.mean(np.abs(err)):.4f} V)')
    plt.grid(True, alpha=0.3)

    plt.tight_layout()


In [3]:
current_dir = Path.cwd()

profiles_dir = current_dir.parent.parent / "Data_preprocessing" / "profiles"
static_parameters_dir = current_dir.parent / "static_parameters"

In [4]:
df_profiles_description =  pd.read_csv(static_parameters_dir / "profiles_description_static_parameters.csv")

In [5]:
df_profiles_description

,bat_num,temp,profile,U_start_V,U_end_V,file_name,Q_Ah,eta,OCV_file_name,SOC_start,SOC_end,SOH
0,1,25,impulse_72_s,4.11093,4.10372,1_+25_impulse_72_s_01.csv,3.289069,0.993163,OCV_1_+25.csv,96.579538,95.849146,93.973411
1,1,25,impulse_144_s,4.10372,4.09365,1_+25_impulse_144_s_02.csv,3.289069,0.993163,OCV_1_+25.csv,95.849146,94.516093,93.973411
2,1,25,impulse_288_s,4.09365,4.07940,1_+25_impulse_288_s_03.csv,3.289069,0.993163,OCV_1_+25.csv,94.516093,91.610379,93.973411
3,1,25,impulse_72_s,4.07940,4.07487,1_+25_impulse_72_s_04.csv,3.289069,0.993163,OCV_1_+25.csv,91.610379,90.357663,93.973411
4,1,25,impulse_144_s,4.07487,4.06139,1_+25_impulse_144_s_05.csv,3.289069,0.993163,OCV_1_+25.csv,90.357663,86.572312,93.973411
...,...,...,...,...,...,...,...,...,...,...,...,...
535,5,35,NEDC,3.95779,3.75069,5_+35_NEDC_02.csv,3.434987,0.991690,OCV_5_+35.csv,76.178929,55.833428,98.142481
536,5,35,NEDC,3.75069,3.59529,5_+35_NEDC_03.csv,3.434987,0.991690,OCV_5_+35.csv,55.833428,36.393882,98.142481
537,5,35,WLTC,4.11297,3.90308,5_+35_WLTC_01.csv,3.434987,0.991690,OCV_5_+35.csv,96.318219,70.142435,98.142481
538,5,35,WLTC,3.90308,3.68861,5_+35_WLTC_02.csv,3.434987,0.991690,OCV_5_+35.csv,70.142435,49.435377,98.142481


In [6]:
# Нужно для каждого профиля достать данные I, U, t из file_name, OCV(SOC) из OCV_file_name

In [7]:
# num_poles = 2
# sp = 600
# sf = sp
# needed_cols = ["R0", "R", "tau", "U_rmse"]

# for col in needed_cols:
#     if col not in df_profiles_description.columns:
#         df_profiles_description[col] = None



# for idx, row in df_profiles_description.iterrows():
#     Q_Ah = row["Q_Ah"]

#     eta = row["eta"]
    
#     z0 = row["SOC_start"]
    
#     df_OCV = pd.read_csv(static_parameters_dir / row["OCV_file_name"])
    
#     df_profile = pd.read_csv(profiles_dir / row["file_name"])
    
#     t = df_profile["t,s"].values
#     I = df_profile["I,A"].values
#     U = df_profile["U,V"].values
    
#     SOC = SOC_func(I, t, z0, Q_Ah, eta)
    
#     OCV = SOC_to_OCV(SOC, df_OCV)
    
#     dynamic_parameters = process_dynamic(eta, Q_Ah, t, I, U, OCV, num_poles, sp, sf)
    
#     U_sim = simulate_battery_model(I, t, OCV, eta, Q_Ah, dynamic_parameters)
    
#     U_rmse = RMSE(U, U_sim)

#     df_profiles_description.loc[idx, "R0"] = dynamic_parameters["R0"]
#     df_profiles_description.loc[idx, "U_rmse"] = U_rmse
    
#     df_profiles_description.at[idx, "R"] = dynamic_parameters["R"].tolist()
#     df_profiles_description.at[idx, "tau"] = dynamic_parameters["tau"].tolist()



    
    

In [ ]:
num_poles = 2
sp_step = 100
sp_min = 100

for col in ["R0", "R", "tau", "U_rmse", "best_sp"]:
    if col not in df_profiles_description.columns:
        df_profiles_description[col] = pd.Series(dtype="object")


for idx, row in df_profiles_description.iterrows():

    Q_Ah = row["Q_Ah"]
    eta = row["eta"]
    z0 = row["SOC_start"]

    df_OCV = pd.read_csv(static_parameters_dir / row["OCV_file_name"])
    df_profile = pd.read_csv(profiles_dir / row["file_name"])

    t = df_profile["t,s"].values
    I = df_profile["I,A"].values
    U = df_profile["U,V"].values

    SOC = SOC_func(I, t, z0, Q_Ah, eta)
    OCV = SOC_to_OCV(SOC, df_OCV)

    N = len(t)

    # 🔹 адаптивная верхняя граница
    sp_max = int(0.45 * N)

    if sp_max < sp_min:
        print(f"Профиль {idx} слишком короткий (N={N})")
        continue

    sp_values = np.arange(sp_min, sp_max + 1, sp_step)

    best_rmse = np.inf
    best_params = None
    best_sp = None

    for sp in sp_values:
        sf = sp

        try:
            dynamic_parameters = process_dynamic(
                eta, Q_Ah, t, I, U, OCV,
                num_poles, sp, sf
            )

            U_sim = simulate_battery_model(
                I, t, OCV, eta, Q_Ah, dynamic_parameters
            )

            rmse = RMSE(U, U_sim)

            if rmse < best_rmse:
                best_rmse = rmse
                best_params = dynamic_parameters
                best_sp = sp

        except Exception:
            continue

    if best_params is not None:
        df_profiles_description.at[idx, "R0"] = best_params["R0"]
        df_profiles_description.at[idx, "R"] = best_params["R"].tolist()
        df_profiles_description.at[idx, "tau"] = best_params["tau"].tolist()
        df_profiles_description.at[idx, "U_rmse"] = best_rmse
        df_profiles_description.at[idx, "best_sp"] = best_sp
        print(idx, best_rmse)

    


0 0.0013228839402132022


In [ ]:
num_poles = 2
sp = 600
sf = sp

row = df_profiles_description.iloc[209]


Q_Ah = row["Q_Ah"]

eta = row["eta"]

z0 = row["SOC_start"]

df_OCV = pd.read_csv(static_parameters_dir / row["OCV_file_name"])

df_profile = pd.read_csv(profiles_dir / row["file_name"])

t = df_profile["t,s"].values
I = df_profile["I,A"].values
U = df_profile["U,V"].values

SOC = SOC_func(I, t, z0, Q_Ah, eta)

OCV = SOC_to_OCV(SOC, df_OCV)

dynamic_parameters = process_dynamic(eta, Q_Ah, t, I, U, OCV, num_poles, sp, sf)

U_sim = simulate_battery_model(I, t, OCV, eta, Q_Ah, dynamic_parameters)

U_rmse = RMSE(U, U_sim)

print(f'RMSE: {U_rmse*1000:.2f} mV')

data_visualization(U, U_sim, OCV, I, t, SOC)

plt.figure(figsize=(6, 4))
plt.plot(df_OCV["SOC, %"], df_OCV['OCV, V'], '-o', color='green', label='OCV(SOC)')
plt.xlabel('SOC')
plt.ylabel('Voltage, V')
plt.title('Open Circuit Voltage vs SOC')
plt.grid(True, alpha=0.3)
plt.legend()

print(dynamic_parameters)

In [ ]:
num_poles = 2
row = df_profiles_description.iloc[1]

Q_Ah = row["Q_Ah"]
eta = row["eta"]
z0 = row["SOC_start"]

df_OCV = pd.read_csv(static_parameters_dir / row["OCV_file_name"])
df_profile = pd.read_csv(profiles_dir / row["file_name"])

t = df_profile["t,s"].values
I = df_profile["I,A"].values
U = df_profile["U,V"].values

SOC = SOC_func(I, t, z0, Q_Ah, eta)
OCV = SOC_to_OCV(SOC, df_OCV)

# Диапазон значений sp для перебора
sp_values = np.arange(200, 1501, 100)  # от 10 до 200 с шагом 10
rmse_values = []

for sp in sp_values:
    sf = sp  # как у вас в коде
    dynamic_parameters = process_dynamic(eta, Q_Ah, t, I, U, OCV, num_poles, sp, sf)
    U_sim = simulate_battery_model(I, t, OCV, eta, Q_Ah, dynamic_parameters)
    
    # RMSE
    rmse = np.sqrt(np.mean((U - U_sim) ** 2))
    rmse_values.append(rmse)

# График
plt.figure(figsize=(8,5))
plt.plot(sp_values, np.array(rmse_values)*1000, marker='o')  # умножаем на 1000, чтобы в mV
plt.xlabel('sp')
plt.ylabel('RMSE [mV]')
plt.title('Зависимость RMSE от sp')
plt.grid(True)

In [ ]:
df_profiles_description

In [ ]:
df_profiles_description.to_csv(current_dir.parent / "dynamic_parameters" / "profiles_description_dynamic_parameters.csv", index=False)